# 02 - Backtest a trained run

Pick a strategy, edit its knobs and the costs, press **Run backtest**. Everything is replayed on the run's own
TEST block (out of sample: the purged split is rebuilt from the run's config) with next-open fills, stops on the
bar's high/low, fees + spread + slippage, and three baselines (buy-and-hold, always-flat, random entries at the
same frequency, holding time and size).

The dashboard stacks price and trades, the per-horizon P(up) against the strategy's entry lines, confidence and
signal strength, predicted sigma, equity and drawdown on one time axis; hover shows every panel at that bar. The
whole block shows the equity story; stops, holding periods and entry-to-exit lines are drawn on views of 800 bars
or fewer, so a detail window follows (its numbers are the window's own). Then the per-trade analytics.

In [1]:
# Parameters
RUN_DIR = None                  # a runs/<id> directory; None -> the newest run under RUNS_DIR
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
DETAIL_BARS = 600
DETAIL_AROUND = "steepest_fall" # "steepest_fall", "worst_trade" or "last"

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from neural_trade.notebook import BacktestExplorer, pick_run
from neural_trade.visualization.trading_dashboard import detail_window

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
explorer = BacktestExplorer.from_run(run_dir, csv_path=CSV_PATH)
display(explorer.widget())
explorer.click_run()   # render the default strategy once; then use the controls

run: ..\runs\20260924T182915Z-1aeff1c-dirty-af67ee43
CalibrationPipeline loaded from '..\runs\20260924T182915Z-1aeff1c-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


Static copy of that first result (the widget above holds the live one): the summary with the random null, the whole block, a detail window, and every trade.

In [3]:
display(explorer.summary_frame(styled=True))
explorer.dashboard().show()
start, end = detail_window(explorer.last, DETAIL_BARS, around=DETAIL_AROUND)
explorer.dashboard(start=start, end=end).show()
explorer.trade_analytics().show()

,calibrated_quantile,buy_and_hold,always_flat,"random same freq (mean of 100 seeds, size 1.00)"
trades,147,1,0,–
net return (after costs),-29.86%,+4.10%,0.00%,-31.53%
gross return (before costs),+2.27%,+4.36%,0.00%,+0.05%
costs paid ($),"3,213",27,0,–
of which fees ($),"2,472",20,0,–
Sharpe after costs (annualised),-84.23,+6.65,0.00,-102.91
max drawdown,30.00%,4.95%,0.00%,–
hit rate after costs,11.6%,100.0%,–,–
hit rate before costs,56.5%,100.0%,–,–
profit factor,0.08,∞,–,–


## Every strategy with its default knobs

Each strategy is compared with random entries at its own trade rate, holding time and position size (the
diamonds; the same null as the summary table above). After costs the return is mostly cost x trade count, so only
the rank against that null tells skill from chance.

In [4]:
runs, comparison = explorer.compare_strategies()
comparison.show()
explorer.comparison_table(runs, styled=True)

,calibrated_quantile,enhanced_multi_horizon,liberal,threshold_spike,buy_and_hold,always_flat
trades,147,0,58,0,1,0
net return (after costs),-29.86%,0.00%,-14.25%,0.00%,+4.10%,0.00%
gross P&L before costs ($),+227,0,-23,0,+436,0
costs paid ($),"3,213",0,"1,402",0,27,0
Sharpe after costs (annualised),-84.23,0.00,-68.27,0.00,+6.65,0.00
max drawdown,30.00%,0.00%,14.25%,0.00%,4.95%,0.00%
hit rate after costs,11.6%,–,0.0%,–,100.0%,–
hit rate before costs,56.5%,–,32.8%,–,100.0%,–
profit factor,0.08,–,0.00,–,∞,–
avg win ($),+14.93,–,–,–,+409.67,–


A strategy with 0 trades is usually blocked by one of two things. Fixed probability lines
(`threshold_spike` enters above 0.65 / below 0.35) are rarely crossed by calibrated probabilities.
Strategies that need the predicted move to agree with the side (`enhanced_multi_horizon`) cannot trade
when delta shrinkage serves a zero delta. Delta shrinkage sets beta = 0 when the raw price head's
moves pointed the wrong way on the calibration block (negative correlation with the realised move).
This run's values:

In [5]:
p_up = explorer.signals.p
pd.DataFrame({"delta beta (served delta = beta x raw)": pd.Series(explorer.blocks["predictor"].bundle.calibration_pipeline.delta_scale),
              "P(up) 1st percentile": np.percentile(p_up, 1, axis=0),
              "P(up) 99th percentile": np.percentile(p_up, 99, axis=0)}, index=["h0", "h1", "h2"]).round(4)

,delta beta (served delta = beta x raw),P(up) 1st percentile,P(up) 99th percentile
h0,0.1234,0.4150,0.5721
h1,0.0000,0.4107,0.6002
h2,0.0695,0.3946,0.5761
